# HW2 — RT-DETR-L (query-based detector)

**Setup (once)** — Colab: Runtime → T4 GPU. Secrets (🔑, left bar, "Notebook access" on): `KAGGLE_USERNAME` + `KAGGLE_KEY`
from kaggle.com → Settings → API → *Create New Token* (or a single `KGAT_...` token as `KAGGLE_API_TOKEN` / `kaggle`);
you must have joined the competition with that account. Checkpoints go to Google Drive and training resumes after a disconnect.
Kaggle — Session options: GPU T4 x2, Internet On, Add data → Competitions → `multiple-object-video-detection-hw-2-ucu-ai-2026`.
Protocol = the YOLO runs: ignore-masked stride-3 train, 6 epochs, seed 0; scored by `coco_utils.score` on all 2 846 val frames.

In [ ]:
!pip install -q -U ultralytics pycocotools

## Environment: locate / download the data, set paths

In [ ]:
import glob, os, pathlib, subprocess
ON_KAGGLE = os.path.exists("/kaggle/input")
if ON_KAGGLE:
    RAW = pathlib.Path(glob.glob("/kaggle/input/**/train/annotations_coco.json", recursive=True)[0]).parents[1]
    WORK, OUT = "/kaggle/working", "/kaggle/working/outputs"
else:  # Colab: data from the Kaggle API, outputs on Drive
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN"):
        try: os.environ[k] = userdata.get(k)
        except Exception: pass
    if "KAGGLE_API_TOKEN" not in os.environ and "KAGGLE_KEY" not in os.environ:
        try: os.environ["KAGGLE_API_TOKEN"] = userdata.get("kaggle")  # one secret named `kaggle` holding the KGAT_... token
        except Exception: pass
    RAW, WORK, OUT = pathlib.Path("/content/raw"), "/content", "/content/drive/MyDrive/hw2_runs"
    if not (RAW / "train/annotations_coco.json").exists():
        subprocess.run("mkdir -p /content/dl /content/raw && cd /content/dl && pip install -q kaggle && "
                       "kaggle competitions download -c multiple-object-video-detection-hw-2-ucu-ai-2026 && "
                       "unzip -q -o *.zip -d /content/raw && rm -f *.zip", shell=True, check=True)
os.makedirs(OUT, exist_ok=True)
os.environ["HW2_DATA"] = str(RAW); os.environ["PYTHONPATH"] = "src"
print("data:", RAW, "| work:", WORK, "| outputs:", OUT)

## Source files (embedded copies of `src/`)

In [ ]:
!mkdir -p src

In [ ]:
%%writefile src/common.py
"""Shared helpers: dataset paths and COCO-json -> DataFrame loading."""
import json
import os
from pathlib import Path

import pandas as pd

# override with HW2_DATA=/path/to/raw when running on another machine
DATA = Path(os.environ.get("HW2_DATA", Path(__file__).resolve().parents[1] / "data" / "raw"))
CLASSES = ["person", "bicycle", "car", "van", "truck", "tricycle", "bus", "motor"]


def load_coco(split):
    """Return (images_df, objects_df, ignore_df) for 'train' or 'val'.

    Ignore regions are stored in the COCO json as iscrowd=1 boxes copied once per
    category (4 regions x 8 classes), so we dedupe them to one row per region.
    objects_df: real objects only (iscrowd=0), category 0..7 (0-based).
    """
    j = json.load(open(DATA / split / "annotations_coco.json"))
    imgs = pd.DataFrame(j["images"]).rename(columns={"id": "image_id"})
    ann = pd.DataFrame(j["annotations"])
    ann[["x", "y", "w", "h"]] = pd.DataFrame(ann.bbox.tolist(), index=ann.index)
    ann = ann.drop(columns=["bbox", "area"]).merge(
        imgs[["image_id", "video", "width", "height"]], on="image_id")
    objs = ann[ann.iscrowd == 0].copy()
    objs["cls"] = objs.category_id - 1
    ign = (ann[ann.iscrowd == 1]
           .drop_duplicates(["image_id", "x", "y", "w", "h"])
           [["image_id", "video", "x", "y", "w", "h", "width", "height"]])
    return imgs, objs, ign

In [ ]:
%%writefile src/coco_utils.py
"""COCO-style scoring shared by every detector family (Ultralytics models in eval_coco.py, torchvision in train_tv.py).

score() = the project's single evaluation protocol: AP (0.5:0.95), AP50, AP75, AP_small/medium/large (COCO bins on
source-pixel area), AR@500, per-class AP and AP without the 4K clip; ignore regions are the iscrowd=1 GT boxes,
maxDets=500 because the Kaggle scorer keeps up to 500 detections per image.
"""
import json
from pathlib import Path

import numpy as np
from pycocotools.cocoeval import COCOeval

from common import CLASSES


def _mean(x):
    return float(np.mean(x[x > -1]))


def score(gt, dets, img_ids, tag, ms_per_frame, out_dir="runs", **meta):
    """dets: list of dict(image_id, category_id (1-based), score, bbox [x, y, w, h] in source pixels)."""
    out = Path(out_dir); out.mkdir(exist_ok=True)
    # raw detections for CPU-only analysis later (analyze.py): image_id, cat, score, x, y, w, h
    np.save(out / f"dets_{tag}.npy", np.array([[d["image_id"], d["category_id"], d["score"], *d["bbox"]] for d in dets], np.float32))
    dt = gt.loadRes(dets)
    E = COCOeval(gt, dt, "bbox")
    E.params.imgIds, E.params.maxDets = img_ids, [1, 10, 500]
    E.evaluate(); E.accumulate(); E.summarize()
    st, P = E.stats, E.eval["precision"][:, :, :, 0, 2]  # P: [iou, recall, class, area=all, maxDets=500]
    st[0] = _mean(P)  # stats[0] is -1 because COCOeval's default summary looks for maxDets=100
    # test has no 4K frames (max 1920x1080) and one 4K val clip is 34% of val frames -> also report AP without it
    E2 = COCOeval(gt, dt, "bbox")
    E2.params.imgIds, E2.params.maxDets = [i for i in img_ids if gt.imgs[i]["width"] < 3000], [1, 10, 500]
    E2.evaluate(); E2.accumulate()
    res = dict(tag=tag, **meta, frames=len(img_ids), AP=st[0], AP_no4k=_mean(E2.eval["precision"][:, :, :, 0, 2]),
               AP50=st[1], AP75=st[2], AP_small=st[3], AP_medium=st[4], AP_large=st[5], AR500=st[8],
               ms_per_frame=ms_per_frame, per_class_AP={n: _mean(P[:, :, k]) for k, n in enumerate(CLASSES)})
    print(json.dumps(res, indent=1))
    json.dump(res, open(out / f"eval_{tag}.json", "w"), indent=1)
    return res

In [ ]:
%%writefile src/prepare.py
"""Build Ultralytics-ready lists + data.yaml (the provided layout images/<video>/N.jpg <-> labels/<video>/N.txt
already matches the YOLO convention, so no label conversion is needed).

- Split: official `train` (56 videos) / `val` (7 videos); no video appears in both, so it is a sequence-level split.
- Frame subsampling: keep every STRIDE-th frame of train (neighbouring frames are near-duplicates).
- Ignore regions are NOT in the YOLO labels; they are only used at evaluation (COCO iscrowd=1, see common.load_coco).

Usage: python src/prepare.py [--stride 3] [--out data/yolo]
"""
import argparse
from pathlib import Path

from common import CLASSES, DATA

ap = argparse.ArgumentParser()
ap.add_argument("--stride", type=int, default=3)
ap.add_argument("--out", default=str(DATA.parent / "yolo"))
a = ap.parse_args()
out = Path(a.out); out.mkdir(parents=True, exist_ok=True)


def frames(split, stride):
    keep = []
    for vid in sorted((DATA / split / "images").iterdir()):
        fr = sorted(vid.glob("*.jpg"))
        keep += fr[::stride]  # same phase for every video -> deterministic
    return keep


tr, va, va4 = frames("train", a.stride), frames("val", 1), frames("val", 4)  # val4: cheap per-epoch curves during training
(out / f"train_s{a.stride}.txt").write_text("\n".join(map(str, tr)))
(out / "val.txt").write_text("\n".join(map(str, va)))
(out / "val_s4.txt").write_text("\n".join(map(str, va4)))
(out / f"data_s{a.stride}.yaml").write_text(
    f"path: {out}\ntrain: train_s{a.stride}.txt\nval: val_s4.txt\nnames:\n"
    + "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASSES)))
print(f"train {len(tr)} frames (stride {a.stride}), val {len(va)} frames -> {out}")

In [ ]:
%%writefile src/mask_ignore.py
"""Make a train set where ignore regions are "don't care": paint them gray and drop labels that sit inside them.

Why: unlabeled crowds/parking lots inside ignore regions would otherwise be learned as background. The scorer does not
penalise detections there (COCO iscrowd=1), so we also should not teach the model to suppress them.
Rule: fill region with gray (114) on the image; drop a label if >50% of its box area is inside a region (~1% of boxes).
Frames without ignore regions are symlinked (no re-encode). Val is left untouched: it is scored with iscrowd (eval_coco.py).

Usage: python src/mask_ignore.py --stride 3      (writes DATA/../yolo_masked/{images,labels,train_sN.txt,data_sN.yaml})
"""
import argparse
from multiprocessing import Pool
from pathlib import Path

from PIL import Image, ImageDraw

from common import CLASSES, DATA

ap = argparse.ArgumentParser()
ap.add_argument("--stride", type=int, default=3)
ap.add_argument("--out", default=str(DATA.parent / "yolo_masked"))
a = ap.parse_args()
OUT = Path(a.out)


def regions(p):  # normalized cx cy w h lines -> list of (x0, y0, x1, y1) normalized
    if not p.exists():
        return []
    return [(cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)
            for cx, cy, w, h in (map(float, l.split()) for l in p.read_text().splitlines() if l.strip())]


def inside_frac(b, r):  # share of box b's area inside region r
    ix = max(0.0, min(b[2], r[2]) - max(b[0], r[0])); iy = max(0.0, min(b[3], r[3]) - max(b[1], r[1]))
    return ix * iy / max((b[2] - b[0]) * (b[3] - b[1]), 1e-12)


def work(img):
    vid, name = img.parent.name, img.stem
    dst_img, dst_lab = OUT / "images" / vid / img.name, OUT / "labels" / vid / f"{name}.txt"
    dst_img.parent.mkdir(parents=True, exist_ok=True); dst_lab.parent.mkdir(parents=True, exist_ok=True)
    regs = regions(DATA / "train" / "ignore" / vid / f"{name}.txt")
    src_lab = DATA / "train" / "labels" / vid / f"{name}.txt"
    lines, kept = (src_lab.read_text().splitlines() if src_lab.exists() else []), []
    for l in lines:
        c, cx, cy, w, h = l.split(); cx, cy, w, h = map(float, (cx, cy, w, h))
        b = (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)
        if not any(inside_frac(b, r) > 0.5 for r in regs):
            kept.append(l)
    dst_lab.write_text("\n".join(kept))
    if dst_img.exists() or dst_img.is_symlink():
        dst_img.unlink()
    if regs:
        im = Image.open(img).convert("RGB"); W, H = im.size; d = ImageDraw.Draw(im)
        for x0, y0, x1, y1 in regs:
            d.rectangle([x0 * W, y0 * H, x1 * W, y1 * H], fill=(114, 114, 114))
        im.save(dst_img, quality=95)
    else:
        dst_img.symlink_to(img)
    return len(lines) - len(kept), bool(regs)


if __name__ == "__main__":
    imgs = []
    for vid in sorted((DATA / "train" / "images").iterdir()):
        imgs += sorted(vid.glob("*.jpg"))[::a.stride]  # same subsampling as prepare.py
    with Pool() as p:
        res = p.map(work, imgs, chunksize=32)
    (OUT / f"train_s{a.stride}.txt").write_text("\n".join(str(OUT / "images" / i.parent.name / i.name) for i in imgs))
    (OUT / f"data_s{a.stride}.yaml").write_text(
        f"path: {OUT}\ntrain: train_s{a.stride}.txt\nval: {Path(a.out).parent / 'yolo' / 'val_s4.txt'}\nnames:\n"
        + "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASSES)))
    print(f"{len(imgs)} frames, {sum(r[1] for r in res)} masked, {sum(r[0] for r in res)} labels dropped -> {OUT}")

In [ ]:
%%writefile src/eval_coco.py
"""COCO-style evaluation of an Ultralytics model on `val` that honours ignore regions (iscrowd=1 in the GT json).

Ultralytics' own val does not know about ignore regions, so false positives inside them are counted against the model;
the Kaggle scorer ignores them. This script is the project's single evaluation protocol for every experiment:
AP (0.5:0.95), AP50, AP75, AP_small/medium/large (COCO bins on source-pixel area), AR@500, per-class AP, speed.

Usage: python src/eval_coco.py --weights runs/x/weights/best.pt --imgsz 1280 --tag smoke
"""
import argparse
import time

from pycocotools.coco import COCO
from ultralytics import RTDETR, YOLO

from coco_utils import score
from common import DATA

ap = argparse.ArgumentParser()
ap.add_argument("--weights", required=True)
ap.add_argument("--imgsz", type=int, default=1280)
ap.add_argument("--conf", type=float, default=0.001)
ap.add_argument("--iou", type=float, default=0.6)     # NMS IoU
ap.add_argument("--split", default="val")
ap.add_argument("--tag", default="eval")
ap.add_argument("--bs", type=int, default=16, help="inference chunk size; lower it for heavy models / small GPUs")
ap.add_argument("--arch", default="yolo", choices=["yolo", "rtdetr"])
ap.add_argument("--limit", type=int, default=0, help="debug: evaluate only the first N frames")
a = ap.parse_args()

gt = COCO(str(DATA / a.split / "annotations_coco.json"))
img_ids = sorted(gt.getImgIds())[: a.limit or None]
files = [str(DATA / a.split / gt.imgs[i]["file_name"]) for i in img_ids]
model, B, dets = (RTDETR if a.arch == "rtdetr" else YOLO)(a.weights), a.bs, []  # chunks of 16: passing every path at once -> CUDA OOM

t0 = time.time()
for s in range(0, len(files), B):
    for r, iid in zip(model.predict(files[s:s + B], imgsz=a.imgsz, conf=a.conf, iou=a.iou, max_det=500, batch=B,
                                    verbose=False), img_ids[s:s + B]):
        xyxy, sc, c = r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy()
        dets += [dict(image_id=iid, category_id=int(k) + 1, score=float(p),
                      bbox=[float(x1), float(y1), float(x2 - x1), float(y2 - y1)])
                 for (x1, y1, x2, y2), p, k in zip(xyxy, sc, c)]
ms = (time.time() - t0) / len(files) * 1000  # end-to-end per frame (decode + preprocess + model + NMS)
score(gt, dets, img_ids, a.tag, ms, weights=a.weights, imgsz=a.imgsz, conf=a.conf, nms_iou=a.iou)

In [ ]:
%%writefile src/train_tv.py
"""Train + evaluate torchvision detectors under the same protocol as the YOLO runs.

  frcnn  : Faster R-CNN R50-FPN v2   (two-stage, anchor-based)
  retina : RetinaNet  R50-FPN v2     (one-stage, anchor-based)
  fcos   : FCOS       R50-FPN        (one-stage, anchor-free)
All start from COCO-pretrained weights (new 8-class heads), train on the YOLO-format lists made by prepare.py /
mask_ignore.py (labels found by replacing /images/ -> /labels/), and are scored on all val frames with coco_utils.score.
Frames are pre-resized so the long side is --max-size (default 1280, like imgsz=1280 for YOLO); boxes are mapped back
to source pixels before scoring.

Usage: python src/train_tv.py --model frcnn --train-list data/yolo_masked/train_s3.txt --tag tv_frcnn
"""
import argparse
import functools
import json
import math
import time
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from pycocotools.coco import COCO
from torch import nn
from torchvision.models import detection as D
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.fcos import FCOSClassificationHead
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
from torchvision.transforms.functional import to_tensor

from coco_utils import score
from common import DATA

ap = argparse.ArgumentParser()
ap.add_argument("--model", required=True, choices=["frcnn", "retina", "fcos"])
ap.add_argument("--train-list", required=True)
ap.add_argument("--tag", required=True)
ap.add_argument("--epochs", type=int, default=6)
ap.add_argument("--bs", type=int, default=4)
ap.add_argument("--lr", type=float, default=0.005)
ap.add_argument("--min-size", type=int, default=720)
ap.add_argument("--max-size", type=int, default=1280)
ap.add_argument("--workers", type=int, default=4)
ap.add_argument("--out", default="runs")
ap.add_argument("--seed", type=int, default=0)
ap.add_argument("--rpn-post-nms", type=int, default=1000, help="frcnn: proposals kept per image after RPN NMS")
ap.add_argument("--roi-batch", type=int, default=512, help="frcnn: RoIs sampled per image for the box head")
ap.add_argument("--viz", type=int, default=0, help="debug: save N augmented training samples with their boxes and exit")
ap.add_argument("--limit-train", type=int, default=0, help="debug: stop after N iterations")
ap.add_argument("--limit-val", type=int, default=0, help="debug: evaluate only the first N frames")
a = ap.parse_args()
torch.manual_seed(a.seed); np.random.seed(a.seed)
dev = "cuda" if torch.cuda.is_available() else "cpu"
NC = 8 + 1  # 8 classes + background (torchvision convention: label 0 = background)


class DS(torch.utils.data.Dataset):
    """Image -> (tensor, target, scale, id). Frames are pre-resized to the working size; hflip only when training."""

    def __init__(s, files, train, ids=None):
        s.files, s.train, s.ids = files, train, ids

    def __len__(s):
        return len(s.files)

    def __getitem__(s, i):
        im = Image.open(s.files[i]).convert("RGB"); W, H = im.size
        sc = min(a.min_size / min(W, H), a.max_size / max(W, H))
        im = im.resize((round(W * sc), round(H * sc)), Image.BILINEAR)
        tgt = {}
        if s.train:
            lab = Path(s.files[i].replace("/images/", "/labels/")).with_suffix(".txt")
            r = np.loadtxt(lab, ndmin=2) if lab.exists() and lab.stat().st_size else np.zeros((0, 5))
            w, h = im.size
            cx, cy, bw, bh = (r[:, 1:5] * np.array([w, h, w, h])).T
            b = np.stack([cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2], 1).reshape(-1, 4)
            keep = ((b[:, 2] - b[:, 0]) >= 1) & ((b[:, 3] - b[:, 1]) >= 1)
            b, c = b[keep], r[keep, 0].astype(np.int64) + 1
            x = to_tensor(im)
            if torch.rand(1).item() < 0.5:  # horizontal flip (torch RNG is re-seeded per dataloader worker, numpy is not)
                x = x.flip(-1); b = b.copy(); b[:, [0, 2]] = w - b[:, [2, 0]]
            tgt = dict(boxes=torch.as_tensor(b, dtype=torch.float32).reshape(-1, 4), labels=torch.as_tensor(c))
            return x, tgt, sc, -1
        return to_tensor(im), tgt, sc, s.ids[i]


def build():
    """COCO-pretrained detector with a fresh 8-class head; inference thresholds set for mAP (low score, up to 500 dets)."""
    size = dict(min_size=(a.min_size - 80, a.min_size - 40, a.min_size + 40, a.min_size + 80, a.min_size),  # multi-scale train; eval uses the last
                max_size=int(a.max_size * 1.12))
    if a.model == "frcnn":
        m = D.fasterrcnn_resnet50_fpn_v2(weights=D.FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1, box_score_thresh=0.001,
                                         box_detections_per_img=500, rpn_post_nms_top_n_train=a.rpn_post_nms,
                                         rpn_post_nms_top_n_test=a.rpn_post_nms, box_batch_size_per_image=a.roi_batch, **size)
        m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, NC)
    elif a.model == "retina":
        m = D.retinanet_resnet50_fpn_v2(weights=D.RetinaNet_ResNet50_FPN_V2_Weights.COCO_V1, score_thresh=0.001,
                                        detections_per_img=500, **size)
        m.head.classification_head = RetinaNetClassificationHead(
            256, m.head.classification_head.num_anchors, NC, norm_layer=functools.partial(nn.GroupNorm, 32))
    else:
        m = D.fcos_resnet50_fpn(weights=D.FCOS_ResNet50_FPN_Weights.COCO_V1, score_thresh=0.001,
                                detections_per_img=500, **size)
        m.head.classification_head = FCOSClassificationHead(256, m.head.classification_head.num_anchors, NC)
    return m.to(dev)


files = Path(a.train_list).read_text().split()
loader = torch.utils.data.DataLoader(DS(files, True), batch_size=a.bs, shuffle=True, num_workers=a.workers, drop_last=True,
                                     collate_fn=lambda b: tuple(zip(*b)), persistent_workers=a.workers > 0)
if a.viz:  # sanity check of labels after resize + flip
    from PIL import ImageDraw
    ds = DS(files, True)
    for k in range(a.viz):
        x, t, _, _ = ds[k * 997 % len(ds)]
        im = Image.fromarray((x.permute(1, 2, 0).numpy() * 255).astype("uint8")); d = ImageDraw.Draw(im)
        for b in t["boxes"].tolist():
            d.rectangle(b, outline=(0, 255, 0), width=2)
        im.save(f"{a.out}/viz_{k}.jpg"); print("saved", k, len(t["boxes"]), "boxes", im.size)
    raise SystemExit
model = build()
opt = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=a.lr, momentum=0.9, weight_decay=1e-4)
total = a.epochs * len(loader)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda i: min(1, (i + 1) / 500) * (0.5 * (1 + math.cos(math.pi * min(i, total) / total))))  # warmup + cosine
scaler = torch.amp.GradScaler(enabled=dev == "cuda")
out = Path(a.out) / a.tag; out.mkdir(parents=True, exist_ok=True)
start, log = 0, []
if (out / "last.pt").exists():  # resume after an interrupted session
    ck = torch.load(out / "last.pt", map_location=dev)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
    start, log = ck["epoch"], ck["log"]
    print("resumed from epoch", start)

it = 0
for ep in range(start, a.epochs):
    model.train(); t0, run = time.time(), []
    for x, tgt, _, _ in loader:
        x = [i.to(dev) for i in x]; tgt = [{k: v.to(dev) for k, v in t.items()} for t in tgt]
        with torch.autocast(dev, enabled=dev == "cuda"):
            losses = model(x, tgt)
            loss = sum(losses.values())
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 10.0)  # RetinaNet/FCOS can spike early with AMP
        scaler.step(opt); scaler.update(); sched.step()
        run.append({k: v.item() for k, v in losses.items()}); it += 1
        if it % 100 == 0:
            print(f"ep {ep + 1}/{a.epochs} it {it} loss {loss.item():.3f} lr {sched.get_last_lr()[0]:.5f} "
                  f"{(time.time() - t0) / len(run):.2f}s/it", flush=True)
        if a.limit_train and it >= a.limit_train:
            break
    log.append(dict(epoch=ep + 1, minutes=(time.time() - t0) / 60, **{k: float(np.mean([r[k] for r in run])) for k in run[0]}))
    print(log[-1], flush=True)
    torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), epoch=ep + 1, log=log), out / "last.pt")
    if a.limit_train and it >= a.limit_train:
        break
json.dump(log, open(out / "train_log.json", "w"), indent=1)

# ---- evaluation on val with the project protocol -------------------------------------------------------------
gt = COCO(str(DATA / "val" / "annotations_coco.json"))
ids = sorted(gt.getImgIds())[: a.limit_val or None]
vl = torch.utils.data.DataLoader(DS([str(DATA / "val" / gt.imgs[i]["file_name"]) for i in ids], False, ids), batch_size=4,
                                 num_workers=a.workers, collate_fn=lambda b: tuple(zip(*b)))
model.eval(); dets, t0 = [], time.time()
with torch.no_grad(), torch.autocast(dev, enabled=dev == "cuda"):
    for x, _, scs, iids in vl:
        for o, sc, iid in zip(model([i.to(dev) for i in x]), scs, iids):
            for (x1, y1, x2, y2), p, c in zip((o["boxes"] / sc).tolist(), o["scores"].tolist(), o["labels"].tolist()):
                dets.append(dict(image_id=iid, category_id=c, score=p, bbox=[x1, y1, x2 - x1, y2 - y1]))
ms = (time.time() - t0) / len(ids) * 1000
score(gt, dets, ids, a.tag, ms, out_dir=a.out, **{k: v for k, v in vars(a).items() if k not in ("train_list", "out", "tag")})

## Stride-3 lists and the ignore-masked train set (local disk, rebuilt in every new session)

In [ ]:
!python src/prepare.py --stride 3 --out {WORK}/yolo
!python src/mask_ignore.py --stride 3 --out {WORK}/yolo_masked

## Train RT-DETR-L (AdamW 1e-4 as is usual for DETR; batch 2 so a 16 GB T4 does not run out of memory)

In [ ]:
from pathlib import Path
from ultralytics import RTDETR
last = Path(OUT) / "rtdetr_l/weights/last.pt"
if last.exists():
    RTDETR(str(last)).train(resume=True)             # continue an interrupted run
else:
    RTDETR("rtdetr-l.pt").train(data=f"{WORK}/yolo_masked/data_s3.yaml", imgsz=1280, epochs=6, batch=2, workers=2,
        optimizer="AdamW", lr0=1e-4, mixup=0.15, scale=0.9, close_mosaic=0, seed=0, deterministic=True,
        project=OUT, name="rtdetr_l", exist_ok=True, plots=False)

## Score on all val frames (COCO AP with ignore regions)

In [ ]:
!python src/eval_coco.py --arch rtdetr --bs 4 --weights {OUT}/rtdetr_l/weights/best.pt --imgsz 1280 --tag rtdetr_l
!cp runs/eval_rtdetr_l.json runs/dets_rtdetr_l.npy {OUT}/
import json, pandas as pd
r = json.load(open("runs/eval_rtdetr_l.json")); print({k: v for k, v in r.items() if k != "per_class_AP"})
d = pd.read_csv(f"{OUT}/rtdetr_l/results.csv"); d.columns = d.columns.str.strip()
print("epochs completed:", len(d)); d.filter(regex="epoch|loss|mAP")